# 09 — End-to-End Model Validation & Portfolio Evaluation

This notebook performs the final offline validation of the core intelligence pipeline before API and frontend development.

It verifies customer-level consistency, CLV quality, segmentation coverage, SHAP coverage, retention recommendations, recommendation performance, and portfolio KPIs.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

print(f"Project root: {PROJECT_ROOT}")

## 1. Load all analytical artifacts

In [ ]:
paths = {
    "features": PROCESSED_DIR / "customer_features.csv",
    "segments": PROCESSED_DIR / "customer_segments.csv",
    "clv": PROCESSED_DIR / "customer_clv_predictions.csv",
    "shap": PROCESSED_DIR / "customer_shap_values.csv",
    "shap_global": PROCESSED_DIR / "clv_shap_global_importance.csv",
    "retention": PROCESSED_DIR / "retention_recommendations.csv",
    "recommendations": PROCESSED_DIR / "recommendations.csv",
    "recommendation_metrics": PROCESSED_DIR / "recommendation_model_comparison.csv",
}

missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing artifacts:\n" + "\n".join(missing)
    )

features = pd.read_csv(paths["features"])
segments = pd.read_csv(paths["segments"])
clv = pd.read_csv(paths["clv"])
shap_values = pd.read_csv(paths["shap"])
shap_global = pd.read_csv(paths["shap_global"])
retention = pd.read_csv(paths["retention"])
recommendations = pd.read_csv(paths["recommendations"])
recommendation_metrics = pd.read_csv(paths["recommendation_metrics"])

print("Features:", features.shape)
print("Segments:", segments.shape)
print("CLV:", clv.shape)
print("SHAP:", shap_values.shape)
print("Retention:", retention.shape)
print("Recommendations:", recommendations.shape)

## 2. Validate customer-level uniqueness

In [ ]:
customer_tables = {
    "features": features,
    "segments": segments,
    "clv": clv,
    "retention": retention,
}

uniqueness_results = []

for name, table in customer_tables.items():
    uniqueness_results.append({
        "artifact": name,
        "rows": len(table),
        "unique_customers": table["customer_unique_id"].nunique(),
        "duplicate_customer_rows": int(
            table["customer_unique_id"].duplicated().sum()
        ),
        "missing_customer_ids": int(
            table["customer_unique_id"].isna().sum()
        ),
    })

uniqueness_df = pd.DataFrame(uniqueness_results)
display(uniqueness_df)

assert (
    uniqueness_df["duplicate_customer_rows"] == 0
).all(), "A customer-level artifact contains duplicate customer rows." 

## 3. Validate cross-artifact customer coverage

In [ ]:
base_customer_ids = set(
    features["customer_unique_id"].astype(str)
)

coverage_rows = []

for name, table in customer_tables.items():
    ids = set(table["customer_unique_id"].astype(str))
    missing_from_features = ids - base_customer_ids

    coverage_rows.append({
        "artifact": name,
        "customers": len(ids),
        "missing_from_feature_table": len(missing_from_features),
        "coverage_pct": (
            100 * len(ids.intersection(base_customer_ids)) / len(ids)
            if ids else 0
        ),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df.round(2))

assert (
    coverage_df["missing_from_feature_table"] == 0
).all(), "A downstream artifact contains unknown customers." 

## 4. Validate CLV prediction quality

In [ ]:
actual = pd.to_numeric(
    clv["future_revenue"], errors="coerce"
).fillna(0)

predicted = pd.to_numeric(
    clv["predicted_future_revenue"], errors="coerce"
).fillna(0)

mae = np.mean(np.abs(actual - predicted))
rmse = np.sqrt(np.mean((actual - predicted) ** 2))

ss_res = np.sum((actual - predicted) ** 2)
ss_tot = np.sum((actual - actual.mean()) ** 2)

r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

clv_metrics = pd.DataFrame({
    "metric": [
        "MAE",
        "RMSE",
        "R2",
        "Mean Actual Future Revenue",
        "Mean Predicted Future Revenue",
        "Total Actual Future Revenue",
        "Total Predicted Future Revenue",
    ],
    "value": [
        mae,
        rmse,
        r2,
        actual.mean(),
        predicted.mean(),
        actual.sum(),
        predicted.sum(),
    ],
})

display(clv_metrics.round(4))

## 5. Check prediction calibration by value band

In [ ]:
calibration = clv[
    [
        "customer_unique_id",
        "future_revenue",
        "predicted_future_revenue",
    ]
].copy()

calibration["predicted_value_band"] = pd.qcut(
    calibration["predicted_future_revenue"].rank(method="first"),
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
)

calibration_summary = (
    calibration
    .groupby("predicted_value_band", observed=False)
    .agg(
        customers=("customer_unique_id", "count"),
        average_actual=("future_revenue", "mean"),
        average_predicted=("predicted_future_revenue", "mean"),
        total_actual=("future_revenue", "sum"),
        total_predicted=("predicted_future_revenue", "sum"),
    )
    .reset_index()
)

display(calibration_summary.round(2))

fig = px.bar(
    calibration_summary,
    x="predicted_value_band",
    y=["average_actual", "average_predicted"],
    barmode="group",
    title="Actual vs Predicted Revenue by Value Band",
)
fig.show()

## 6. Validate segmentation coverage and business value

In [ ]:
segment_summary = (
    segments
    .groupby("segment", dropna=False)
    .agg(
        customers=("customer_unique_id", "count"),
        average_revenue=("total_revenue", "mean"),
        total_revenue=("total_revenue", "sum"),
        average_recency=("recency_days", "mean"),
        average_orders=("order_count", "mean"),
    )
    .reset_index()
)

segment_summary["customer_share_pct"] = (
    100 * segment_summary["customers"] / segment_summary["customers"].sum()
)

segment_summary["revenue_share_pct"] = (
    100 * segment_summary["total_revenue"] / segment_summary["total_revenue"].sum()
)

display(segment_summary.round(2))

fig = px.bar(
    segment_summary.sort_values("total_revenue"),
    x="total_revenue",
    y="segment",
    orientation="h",
    title="Revenue by Customer Segment",
)
fig.show()

## 7. Validate SHAP coverage

In [ ]:
shap_columns = [
    column for column in shap_values.columns
    if column.startswith("shap_")
]

shap_validation = pd.DataFrame({
    "shap_customer_rows": [len(shap_values)],
    "unique_customers": [shap_values["customer_unique_id"].nunique()],
    "shap_feature_columns": [len(shap_columns)],
    "missing_shap_values": [
        int(shap_values[shap_columns].isna().sum().sum())
    ],
})

display(shap_validation)

assert (
    shap_validation.loc[0, "missing_shap_values"] == 0
), "SHAP table contains missing contribution values." 

## 8. Validate retention recommendations

In [ ]:
required_retention_columns = [
    "customer_unique_id",
    "retention_priority",
    "recommended_action",
    "recommendation_reason",
    "retention_strategy",
    "opportunity_score",
]

missing_retention = [
    column for column in required_retention_columns
    if column not in retention.columns
]

if missing_retention:
    raise ValueError(
        f"Missing retention columns: {missing_retention}"
    )

retention_completeness = pd.DataFrame({
    "field": required_retention_columns,
    "missing_values": [
        int(retention[column].isna().sum())
        for column in required_retention_columns
    ],
})

display(retention_completeness)

assert (
    retention_completeness["missing_values"] == 0
).all(), "Retention recommendations contain missing required fields." 

## 9. Identify high-value retention opportunities

In [ ]:
high_value_cutoff = retention[
    "predicted_future_revenue"
].quantile(0.75)

high_recency_cutoff = retention[
    "recency_days"
].quantile(0.75)

opportunity = retention[
    (retention["predicted_future_revenue"] >= high_value_cutoff)
    & (retention["recency_days"] >= high_recency_cutoff)
].copy()

opportunity_summary = pd.DataFrame({
    "metric": [
        "High-value inactive customers",
        "Predicted revenue at opportunity",
        "Average predicted value",
        "Average recency",
    ],
    "value": [
        len(opportunity),
        opportunity["predicted_future_revenue"].sum(),
        opportunity["predicted_future_revenue"].mean()
        if not opportunity.empty else 0,
        opportunity["recency_days"].mean()
        if not opportunity.empty else 0,
    ],
})

display(opportunity_summary.round(2))

if not opportunity.empty:
    display(
        opportunity[
            [
                "customer_unique_id",
                "predicted_future_revenue",
                "recency_days",
                "retention_priority",
                "recommended_action",
                "opportunity_score",
            ]
        ]
        .sort_values("predicted_future_revenue", ascending=False)
        .head(20)
    )

## 10. Evaluate recommendation-system performance

In [ ]:
display(recommendation_metrics.round(4))

metric_columns = [
    column for column in [
        "precision_at_k",
        "recall_at_k",
        "hit_rate_at_k",
    ]
    if column in recommendation_metrics.columns
]

if metric_columns:
    comparison_plot = recommendation_metrics.melt(
        id_vars=["method"],
        value_vars=metric_columns,
        var_name="metric",
        value_name="score",
    )

    fig = px.bar(
        comparison_plot,
        x="metric",
        y="score",
        color="method",
        barmode="group",
        title="Recommendation Performance",
    )
    fig.show()

## 11. Calculate portfolio-level KPIs

In [ ]:
kpis = {
    "Customers": int(features["customer_unique_id"].nunique()),
    "Predicted Future Revenue": float(
        retention["predicted_future_revenue"].sum()
    ),
    "Average Predicted Customer Value": float(
        retention["predicted_future_revenue"].mean()
    ),
    "Critical Retention Customers": int(
        (retention["retention_priority"] == "Critical").sum()
    ),
    "High Retention Customers": int(
        (retention["retention_priority"] == "High").sum()
    ),
    "High-Value Inactive Customers": int(len(opportunity)),
    "Recommendation Rows": int(len(recommendations)),
}

kpi_df = pd.DataFrame({
    "KPI": list(kpis.keys()),
    "Value": list(kpis.values()),
})

display(kpi_df)

## 12. Save portfolio evaluation report

In [ ]:
REPORT_PATH = PROCESSED_DIR / "portfolio_evaluation_report.csv"

report_rows = []

for _, row in clv_metrics.iterrows():
    report_rows.append({
        "category": "CLV",
        "metric": row["metric"],
        "value": row["value"],
    })

for _, row in opportunity_summary.iterrows():
    report_rows.append({
        "category": "Retention Opportunity",
        "metric": row["metric"],
        "value": row["value"],
    })

for key, value in kpis.items():
    report_rows.append({
        "category": "Portfolio KPI",
        "metric": key,
        "value": value,
    })

evaluation_report = pd.DataFrame(report_rows)

evaluation_report.to_csv(
    REPORT_PATH,
    index=False,
)

print(f"Saved portfolio evaluation report: {REPORT_PATH}")

# Final validation checklist

The analytical layer is ready for application development when:

- Customer feature table exists
- Segmentation artifact exists
- CLV model exists
- SHAP explanations exist
- Retention recommendations exist
- Recommendation evaluation exists
- Cross-artifact customer consistency passes
- Required recommendation fields are complete
- Portfolio KPIs are generated

Expected new artifact:

```text
data/processed/portfolio_evaluation_report.csv
```

The next stage moves into the FastAPI application layer.